# 🚀 混合式 RAG 結合 Microsoft phi-4-mini 及 SQLite FTS5（零雲端 SLM）

> **作者：** Çağrı Giray Keşan ([@Cagrik34](https://github.com/Cagrik34))  
> **焦點：** 小型語言模型（SLM）、SQLite FTS5 BM25、密集嵌入、相互排序融合（RRF）

---

## 📌 1. 動機：本地 SLM 中的關鍵字召回兩難
僅依賴密集向量嵌入的標準 RAG 架構常常無法準確檢索數值型標記（例如 `2,340,000 TL`、合約編號、帳戶號碼）。  
反之，稀疏詞彙搜尋（BM25）則會遺漏語義同義詞和同義改寫的問題。

本手冊示範如何實作一個<strong>高速、記憶體內混合檢索引擎</strong>，結合：
1. <strong>密集向量</strong>（餘弦相似度）
2. <strong>稀疏詞彙搜尋</strong>（SQLite FTS5 BM25）
3. **相互排序融合（RRF, $k=60$）**
4. 使用 Microsoft `phi-4-mini` 進行 **具依據的引文生成（`[1]`、`[2]`）**。


In [ ]:
import os
import sqlite3
import numpy as np
from typing import List, Tuple, Dict, Any

RRF_K = 60
TOP_K = 2
print("✅ Core dependencies loaded successfully.")

## 🏗️ 2. 雙重 SQLite 架構（密集向量 + 虛擬 FTS5 BM25 表）


In [ ]:
class LocalHybridRAGStore:
    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self) -> None:
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS document_chunks (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_file TEXT NOT NULL,
                    chunk_index INTEGER NOT NULL,
                    content TEXT NOT NULL,
                    embedding BLOB NOT NULL
                )
            """)
            self.conn.execute("""
                CREATE VIRTUAL TABLE IF NOT EXISTS document_chunks_fts USING fts5(
                    content,
                    source_file UNINDEXED,
                    chunk_index UNINDEXED,
                    tokenize='unicode61'
                )
            """)

    def insert_chunk(self, source_file: str, chunk_index: int, content: str, embedding: List[float]) -> None:
        vec = np.array(embedding, dtype=np.float32)
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm

        with self.conn:
            self.conn.execute(
                "INSERT INTO document_chunks (source_file, chunk_index, content, embedding) VALUES (?, ?, ?, ?)",
                (source_file, chunk_index, content, vec.tobytes())
            )
            self.conn.execute(
                "INSERT INTO document_chunks_fts (content, source_file, chunk_index) VALUES (?, ?, ?)",
                (content, source_file, str(chunk_index))
            )

    def search_dense(self, query_embedding: List[float], top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        q_vec = np.array(query_embedding, dtype=np.float32)
        q_norm = np.linalg.norm(q_vec)
        if q_norm > 0:
            q_vec = q_vec / q_norm

        cursor = self.conn.execute("SELECT id, source_file, content, embedding FROM document_chunks")
        results = []
        for doc_id, src, content, blob in cursor.fetchall():
            doc_vec = np.frombuffer(blob, dtype=np.float32)
            similarity = float(np.dot(q_vec, doc_vec))
            results.append((doc_id, src, content, similarity))
        results.sort(key=lambda x: x[3], reverse=True)
        return results[:top_k]

    def search_sparse_bm25(self, query_text: str, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        clean_tokens = [t for t in query_text.replace("'", "").replace('"', '').split() if len(t) > 1]
        if not clean_tokens:
            return []
        fts_query = " OR ".join(f'"{t}"' for t in clean_tokens)
        cursor = self.conn.execute(
            "SELECT rowid, source_file, content, rank FROM document_chunks_fts WHERE document_chunks_fts MATCH ? ORDER BY rank LIMIT ?",
            (fts_query, top_k)
        )
        results = []
        for doc_id, src, content, bm25_rank in cursor.fetchall():
            bm25_score = 1.0 / (1.0 + abs(float(bm25_rank)))
            results.append((doc_id, src, content, bm25_score))
        return results

    def hybrid_search(self, query_text: str, query_embedding: List[float], top_k: int = TOP_K) -> List[Dict[str, Any]]:
        dense_hits = self.search_dense(query_embedding, top_k=10)
        sparse_hits = self.search_sparse_bm25(query_text, top_k=10)
        fused_scores = {}
        chunk_map = {}

        for rank, (doc_id, src, content, sim) in enumerate(dense_hits, start=1):
            key = f"{src}::{content[:50]}"
            chunk_map[key] = (src, content, "vector")
            fused_scores[key] = fused_scores.get(key, 0.0) + (1.0 / (RRF_K + rank))

        for rank, (doc_id, src, content, bm25) in enumerate(sparse_hits, start=1):
            key = f"{src}::{content[:50]}"
            if key not in chunk_map:
                chunk_map[key] = (src, content, "bm25")
            else:
                chunk_map[key] = (src, content, "hybrid")
            fused_scores[key] = fused_scores.get(key, 0.0) + (1.0 / (RRF_K + rank))

        sorted_keys = sorted(fused_scores.keys(), key=lambda k: fused_scores[k], reverse=True)[:top_k]
        output = []
        for citation_idx, key in enumerate(sorted_keys, start=1):
            src, content, match_type = chunk_map[key]
            output.append({
                "citation_index": citation_idx,
                "source_file": src,
                "content": content,
                "rrf_score": fused_scores[key],
                "match_type": match_type
            })
        return output

print("✅ LocalHybridRAGStore class compiled successfully.")

## 📊 3. 樣本攝取及執行基準測試


In [ ]:
store = LocalHybridRAGStore()

sample_docs = [
    ("q3_financial_report.pdf", 0, "CodePulse engineering project total Q3 budget was allocated at 2,340,000 TL with 15 active developers.", [0.8, 0.1, 0.2] + [0.0] * 1021),
    ("architecture_specs.md", 0, "Zenith AI leverages Microsoft phi-4-mini (3.8B parameters) for local zero-cloud inference.", [0.2, 0.9, 0.1] + [0.0] * 1021),
    ("hr_policy_2026.docx", 0, "Remote work expense allowance is capped at 15,000 TL per employee quarterly.", [0.1, 0.1, 0.8] + [0.0] * 1021)
]

for src, idx, content, emb in sample_docs:
    store.insert_chunk(src, idx, content, emb)

query = "What is the total allocated budget for the CodePulse project?"
query_vec = [0.75, 0.15, 0.25] + [0.0] * 1021

results = store.hybrid_search(query, query_vec, top_k=2)
for res in results:
    print(f"[{res['citation_index']}] {res['source_file']} ({res['match_type'].upper()}) -> Score: {res['rrf_score']:.4f}")
    print(f"    Content: {res['content']}\n")

## 📝 4. 為 Microsoft phi-4-mini 制定有根據的提示語


In [ ]:
def construct_grounded_prompt(query: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    context_blocks = []
    for chunk in retrieved_chunks:
        context_blocks.append(f"[{chunk['citation_index']}] (Source: {chunk['source_file']})\n{chunk['content']}")
    context_str = "\n\n".join(context_blocks)

    return f"""You are Zenith AI, an enterprise-grade local assistant.
Answer the user query strictly based on the provided context below.
Every factual claim must cite its source index like [1] or [2].
If the context does not contain the answer, respond: 'This information is not present in the indexed documents.'

--- CONTEXT ---
{context_str}
--- END CONTEXT ---

User Query: {query}
Answer:"""

prompt = construct_grounded_prompt(query, results)
print(prompt)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
